### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

    1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
    2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")
response=model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They’re birds, right? Some species like the African Grey Parrot are known for their ability to mimic human speech. But why do they do that? Is it just for communication, or is there a deeper reason?\n\nFirst, I remember that parrots are social animals. They live in groups in the wild. Maybe talking is a way to interact with each other? Like how some animals use calls to communicate. But humans use words, so maybe parrots are imitating us to fit in? Or perhaps they learn from their environment.\n\nWait, I\'ve heard that parrots have a part of their brain similar to humans involved in language processing. The neostriatum? That might help them mimic sounds. But why would natural selection favor this trait? Maybe because mimicking helps them in the wild. Like, if they can copy other birds or animals, it might help them avoid predators or find food. But I\'m not s

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunnny in {location}"

model_with_tools=model.bind_tools([get_weather])


In [4]:
response = model_with_tools.invoke("What's the weather like in Booston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Booston. First, I need to check if "Booston" is the correct spelling. I think they might have meant Boston, Massachusetts. Since the function requires a location parameter, I should use the get_weather function with the location set to Boston. I\'ll proceed to call the function with that parameter.\n', 'tool_calls': [{'id': '18wm0z142', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 155, 'total_tokens': 251, 'completion_time': 0.190023107, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt_time': 0.006068827, 'prompt_tokens_details': None, 'queue_time': 0.161756632, 'total_time': 0.196091934}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provid

### Tool Execution Loops

In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)


# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny ☀️. Let me know if you need more details!


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'hs21a0s5b', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.137795955, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.007111652, 'prompt_tokens_details': None, 'queue_time': 0.166011977, 'total_time': 0.144907607}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_deman